# Existence equation — panel logit

Whether a cross-pool arbitrage exists. The **onset** risk set (`gap_lag == 0`): given no gap is open,
does one appear at `t`?

$$\Pr\!\big(D_{p,t}=1 \mid X_{p,t-1}\big)=\Lambda(\eta_{p,t}),\qquad \Lambda(z)=\frac{1}{1+e^{-z}}$$

$$
\eta_{p,t}=\alpha_p
+\beta_2\,\log(\text{base\_fee}_t)+\beta_3\,\text{gas\_util}_{t-1}+\beta_4\,\log(1+\text{tip\_p90}_{t-1})
+\beta_5\,\overline{\log(1+\text{mev})}_{p,t-1}+\beta_6\,\overline{\text{nb\_swaps\_ewma}}_{p,t-1}
+\beta_7\,\log(\text{ewma\_vol}_t)
+\gamma_{h(t)}
$$

`base_fee` and `ewma_vol` enter at `t`; every other regressor is lagged one block. $\alpha_p$ are
pool-pair and $\gamma_{h(t)}$ hour-of-day fixed effects. All logic lives in `arblib.estimation`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

from arblib import estimation as est
from arblib.config import STUDY as S

panel = est.load_panel(S)
print("panel:", panel.shape)

panel: (430420, 27)


## Onset risk set

Rows with `gap_lag == 0`, dropping pairs with no `D` variation (uninformative under pair FE).

In [2]:
onset = est.build_risk_set(panel, quantile=0.2, condition="onset")
# EPV floor: drop pairs whose rarer outcome class (here usually the onset events D=1) is too thin


onset: (405973, 15) | D mean: 0.025 | pairs: 10


In [3]:
onset.groupby("pair").size()

pair
pancake_1_vs_pancake_2    40794
uniswap_1_vs_pancake_1    42908
uniswap_1_vs_pancake_2    40616
uniswap_1_vs_uniswap_2    41830
uniswap_1_vs_uniswap_3    43023
uniswap_2_vs_pancake_1    41258
uniswap_2_vs_pancake_2    26683
uniswap_2_vs_uniswap_3    42948
uniswap_3_vs_pancake_1    43022
uniswap_3_vs_pancake_2    42891
dtype: int64

In [4]:
onset = est.drop_pairs_by_event_floor(onset, S.min_events_per_pair, label="onset")

onset: dropping 2 pairs with < 55 rarer-class events (min(n_D0, n_D1)): ['uniswap_1_vs_uniswap_3', 'uniswap_3_vs_pancake_1']
onset: kept 8 pairs, 319928 rows


In [5]:
onset["D"].mean()

np.float64(0.03172901402815634)

## Fit — logit & probit

Pool-pair fixed effects `C(pair)`, cluster-robust SEs by pair. Covariates are mean-centred so the
intercept is read at an average observation.

In [6]:
onset_c = est.center_continuous(onset, est.ONSET_TERMS)
res_logit  = est.fit_hazard_logit(onset_c, est.ONSET_TERMS, direction="logit")
res_probit = est.fit_hazard_logit(onset_c, est.ONSET_TERMS, direction="probit")
print(res_logit.summary())
#print(res_probit.summary())

                           Logit Regression Results                           
Dep. Variable:                      D   No. Observations:               319928
Model:                          Logit   Df Residuals:                   319891
Method:                           MLE   Df Model:                           36
Date:                Sun, 23 Aug 2026   Pseudo R-squ.:                  0.2567
Time:                        17:35:39   Log-Likelihood:                -33461.
converged:                       True   LL-Null:                       -45014.
Covariance Type:              cluster   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -3.6466      0.045    -80.978      0.000      -3.735      -3.558
C(pair)[T.uniswap_1_vs_pancake_1]    -2.7182      0.047    -57

In [7]:
print(res_probit.summary())

                          Probit Regression Results                           
Dep. Variable:                      D   No. Observations:               319928
Model:                         Probit   Df Residuals:                   319891
Method:                           MLE   Df Model:                           36
Date:                Sun, 23 Aug 2026   Pseudo R-squ.:                  0.2583
Time:                        17:35:39   Log-Likelihood:                -33385.
converged:                       True   LL-Null:                       -45014.
Covariance Type:              cluster   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -1.9276      0.013   -150.197      0.000      -1.953      -1.902
C(pair)[T.uniswap_1_vs_pancake_1]    -1.0662      0.020    -53

## Separation diagnostic — where is it concentrated?

If statsmodels flags quasi-separation, this shows where it sits: the share of near-perfectly predicted
observations, how they split across `pair` / `hour` cells, and the fixed-effect dummies with the
classic signature (extreme coefficient, tight SE). A concentration in a few sparse cells leaves the
economic covariates identified; a share spread everywhere would be a real problem.

In [8]:
est.separation_report(res_logit, onset_c)



perfectly predicted (fitted p < 0.0001 or > 0.9999): 0 / 319928 obs (0.0%)

fixed-effect dummies by |z| (separation signature = big |coef|, tiny SE), top 8:
                                    coef  std_err   abs_z
C(pair)[T.uniswap_3_vs_pancake_2] -2.299    0.023  99.938
C(pair)[T.uniswap_1_vs_pancake_1] -2.718    0.047  57.794
C(pair)[T.uniswap_2_vs_uniswap_3] -3.183    0.062  51.754
C(pair)[T.uniswap_2_vs_pancake_2]  1.897    0.040  47.942
C(pair)[T.uniswap_1_vs_uniswap_2] -0.900    0.106   8.471
C(hour)[T.13]                      0.219    0.028   7.700
C(pair)[T.uniswap_2_vs_pancake_1] -0.419    0.075   5.559
C(hour)[T.14]                      0.173    0.042   4.133


## Multicollinearity check

VIFs (worry above ~10) and the covariate correlation matrix.

In [10]:
print(est.variance_inflation(onset, est.ONSET_TERMS))
onset[est.ONSET_TERMS].corr().round(3)

const           2359.824561
log_base_fee       1.145460
gas_util_lag       1.056612
tip_p90_lag        1.119816
mev_lag            1.590098
freq_lag           1.408026
log_vol            1.507132
Name: VIF, dtype: float64


,log_base_fee,gas_util_lag,tip_p90_lag,mev_lag,freq_lag,log_vol
log_base_fee,1.000,0.037,0.022,0.163,0.047,0.340
gas_util_lag,0.037,1.000,-0.221,0.008,0.024,-0.002
tip_p90_lag,0.022,-0.221,1.000,0.174,0.130,0.222
mev_lag,0.163,0.008,0.174,1.000,0.518,0.478
freq_lag,0.047,0.024,0.130,0.518,1.000,0.353
log_vol,0.340,-0.002,0.222,0.478,0.353,1.000
